# 🇸🇾🇱🇧🇯🇴🇵🇸 ShamiBERT: A BERT Model for the Levantine Dialect

## Levantine Arabic BERT - Continual Pre-training

This notebook builds the **ShamiBERT** model by performing **Continual Pre-training** on the **AraBERT-Twitter** model using **Levantine Arabic dialect data** (Syria, Lebanon, Jordan, and Palestine).

### Methodology (Similar to SaudiBERT)

- **Base Model**: `aubmindlab/bert-base-arabertv02-twitter`
  - Pretrained on **77GB of Arabic text** + **60M Arabic tweets**
  - Supports **emojis and dialectal content**
  - A strong starting point for **Levantine dialect data**

- **Training Task**: **Masked Language Modeling (MLM)**

- **Data**: A collection of available **Levantine dialect datasets**

- **Acceleration**: **Unsloth** is used to **speed up training and reduce memory usage**

## 1. Install Libraries

## 2. Check settings and GPU

In [3]:
import torch
import os
import gc

# Check GPU
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

# Configuration
MODEL_NAME = "aubmindlab/bert-base-arabertv02-twitter"
OUTPUT_MODEL_NAME = "ShamiBERT"  # سيتم تغييره عند الرفع
MAX_SEQ_LENGTH = 128  # AraBERT-Twitter مدرب على 64، نرفعها لـ 128
BATCH_SIZE = 32
LEARNING_RATE = 2e-5
NUM_EPOCHS = 5
MLM_PROBABILITY = 0.15
WARMUP_RATIO = 0.1
SEED = 42

print(f"\n=== ShamiBERT Configuration ===")
print(f"Base Model: {MODEL_NAME}")
print(f"Max Seq Length: {MAX_SEQ_LENGTH}")
print(f"Batch Size: {BATCH_SIZE}")
print(f"Learning Rate: {LEARNING_RATE}")
print(f"Epochs: {NUM_EPOCHS}")
print(f"MLM Probability: {MLM_PROBABILITY}")

PyTorch version: 2.10.0+cu128
CUDA available: True
GPU: NVIDIA A100-SXM4-40GB
VRAM: 42.4 GB

=== ShamiBERT Configuration ===
Base Model: aubmindlab/bert-base-arabertv02-twitter
Max Seq Length: 128
Batch Size: 32
Learning Rate: 2e-05
Epochs: 5
MLM Probability: 0.15


## 3. Load Base Modal And Tokenizer

## 4. Intilaize Levantine Dataset
We will collect from diffrent resources

In [6]:
import re
import pandas as pd
from datasets import load_dataset, Dataset, concatenate_datasets
from arabert.preprocess import ArabertPreprocessor

# Initialize AraBERT preprocessor (Twitter variant)
arabert_prep = ArabertPreprocessor(
    model_name="bert-base-arabertv02-twitter",
    keep_emojis=True
)

def clean_text(text):
    """Clean and preprocess Levantine Arabic text"""
    if not isinstance(text, str) or len(text.strip()) < 5:
        return None

    # Remove URLs
    text = re.sub(r'https?://\S+|www\.\S+', '', text)
    # Remove HTML tags
    text = re.sub(r'<[^>]+>', '', text)
    # Remove excessive whitespace
    text = re.sub(r'\s+', ' ', text).strip()
    # Remove very short texts
    if len(text.split()) < 3:
        return None

    # Apply AraBERT preprocessing
    try:
        text = arabert_prep.preprocess(text)
    except:
        pass

    return text if len(text.strip()) > 5 else None

all_texts = []
print("=== جمع البيانات الشامية ===")

=== جمع البيانات الشامية ===


In [ ]:
# ============================================
# Dataset 1: QCRI Arabic POS Dialect (Levantine subset)
# ============================================
print("\n📦 Loading QCRI Arabic POS Dialect (Levantine)...")

try:
    ds_qcri = load_dataset("QCRI/arabic_pos_dialect", "lev", split="train")

    qcri_texts = []

    for item in ds_qcri:

        words = item.get("words", [])

        # remove special tokens
        words = [w for w in words if w not in ["EOS", "-"]]

        sentence = " ".join(words)

        cleaned = clean_text(sentence)

        if cleaned:
            qcri_texts.append(cleaned)

    all_texts.extend(qcri_texts)

    print(f"  ✅ QCRI LEV: {len(qcri_texts)} texts")

except Exception as e:
    print(f"  ⚠️ QCRI: {e}")


📦 Loading QCRI Arabic POS Dialect (Levantine)...
  ✅ QCRI LEV: 350 texts


In [21]:
print("\n📦 Loading Levanti dataset...")

try:
    ds_levanti = load_dataset("guymorlan/levanti", split="train", streaming=True)

    levanti_texts = []

    for item in ds_levanti:
        if 'arabic' in item and item['arabic']:
            cleaned = clean_text(str(item['arabic']))
            if cleaned:
                levanti_texts.append(cleaned)

    all_texts.extend(levanti_texts)
    print(f"  ✅ Levanti: {len(levanti_texts)} texts")

except Exception as e:
    print(f"  ⚠️ Levanti: {e}")


📦 Loading Levanti dataset...
  ✅ Levanti: 505480 texts


In [ ]:
# ============================================
# Dataset 2: Generate Synthetic Levantine Data
# Common Shami phrases and expressions for augmentation
# ============================================
print("\n📦 Adding curated Levantine expressions...")

# مجموعة من العبارات والجمل الشامية الشائعة
shami_phrases = [
    # تحيات وعبارات يومية
    "كيفك انشالله تكون بخير",
    "شو أخبارك يا زلمة",
    "الله يعطيك العافية",
    "تسلم إيدك على الأكل",
    "يسلمو دياتك",
    "مرحبا كيف الحال",
    "أهلا وسهلا فيك",
    "الحمدلله مناح",
    "شو عملت اليوم",
    "وين رحت مبارح",
    # حياة يومية
    "هلأ رايح عالشغل بالباص",
    "بدي روح عالسوق أشتري خضرا",
    "الجو كتير حلو اليوم",
    "الطريق مسكرة بسبب الزحمة",
    "ما عندي وقت هلأ",
    "خلص الشغل بكرا إن شاء الله",
    "الأكل عند أمي أطيب شي",
    "بحب الفلافل والحمص",
    "شو رأيك نروح عالمطعم",
    "الشباب كلهم رايحين عالملعب",
    # تعبيرات مختلفة
    "يا الله شو هالحكي الحلو",
    "ما في مشكلة خلص",
    "طيب ماشي الحال",
    "لك شو هالقصة",
    "والله ما بعرف شو صار",
    "يا ريت كان عنا وقت أكتر",
    "هاد الشي كتير منيح",
    "بتعرف وين السوبرماركت",
    "خليني فكر شوي",
    "ما حدا بيعرف الجواب",
    # تعبيرات سورية
    "يا خيي تعال نتعشى سوا",
    "شو هالأكلة الطيبة",
    "لك روح نام أحسنلك",
    "بدك شي من الدكانة",
    "هي المدرسة قريبة من البيت",
    # تعبيرات لبنانية
    "كيفك حبيبي شو عم تعمل",
    "يلا نروح عالبحر",
    "هيدا الشي كتير حلو",
    "ما بقا فيي استحمل",
    "شو حلوة هالبنت",
    # تعبيرات أردنية
    "يا زلمة وين كنت",
    "تعال اقعد معنا",
    "هسا بنزل عالسوق",
    "اشي كتير زاكي",
    "ما بدي أحكي عن هالموضوع",
    # تعبيرات فلسطينية
    "ليش ما جيت عالحفلة",
    "قديش حقو هاد",
    "يا ريت تيجي بكرا",
    "وين أخوك ما شفتو من زمان",
    "إسا بنوصل إن شاء الله",
    # جمل أطول
    "رحت عالمطعم اللي بوسط البلد وكان الأكل كتير طيب والخدمة ممتازة",
    "مبارح كنا عند خالتي وعملتلنا أكل شامي على أصولو",
    "الدكتور قلي لازم ارتاح أسبوع وما أشتغل",
    "البنات رايحين عالجامعة الصبح وراجعين بعد الضهر",
    "هلأ الأسعار كتير غالية وما حدا قادر يعيش",
    "شو رأيك نعمل رحلة عالجبل هالويكند",
    "أنا بحب الشتا بس ما بحب البرد الكتير",
    "الولاد بالمدرسة وأنا قاعد بالبيت لحالي",
    "كل يوم نفس الروتين وبدي أغير شي بحياتي",
    "الحارة كلها عم تحكي عن اللي صار مبارح",
    "عنا عرس الأسبوع الجاي ولسا ما جهزنا شي",
    "الطقس بردان كتير وما بدي أطلع من البيت",
    "خلصت الامتحانات والحمدلله نجحت بكلشي",
    "بدنا نسافر عالصيف بس لسا ما حجزنا",
    "الجيران كتير منيحين وبيساعدونا بكلشي",
]

# Clean and add
curated_texts = [clean_text(t) for t in shami_phrases]
curated_texts = [t for t in curated_texts if t]
all_texts.extend(curated_texts)
print(f"  ✅ Curated Shami: {len(curated_texts)} texts")


📦 Adding curated Levantine expressions...
  ✅ Curated Shami: 63 texts


In [23]:
# ============================================
# Deduplicate and prepare final dataset
# ============================================
print("\n" + "="*50)
print("=== إعداد البيانات النهائية ===")
print("="*50)

# Remove duplicates
all_texts = list(set(all_texts))

# Remove None and empty strings
all_texts = [t for t in all_texts if t and len(t.strip()) > 10]

print(f"\n📊 Total unique Levantine texts: {len(all_texts)}")
print(f"📊 Average text length: {sum(len(t) for t in all_texts) / max(len(all_texts), 1):.0f} chars")

# Show samples
print("\n📝 عينات من البيانات:")
import random
random.seed(SEED)
for i, text in enumerate(random.sample(all_texts, min(5, len(all_texts)))):
    print(f"  [{i+1}] {text[:100]}..." if len(text) > 100 else f"  [{i+1}] {text}")


=== إعداد البيانات النهائية ===

📊 Total unique Levantine texts: 504565
📊 Average text length: 58 chars

📝 عينات من البيانات:
  [1] الشركة بتدهلز للسوق الجديد قبل ما تطرح منتجاتها فيه .
  [2] إذا عندك تلات بنات وشاب بإيدك ، فينك تعمل فل هاوس وتربح الجولة
  [3] الدكتور قال إنه ما في أمل يصح ، بس إحنا ما زلنا متفائلين .
  [4] خد نفس عميء وحاول تتعامل مع المشكلة دي برواءة
  [5] يا ريت تخبرني بكرا إذا بتقدر تيجي ع الحفلة . عم إستنى أخبارك بفارغ الصبر .


In [24]:
# ============================================
# Data Augmentation: Repeat small dataset for more training signal
# (مشابه لمنهجية SaudiBERT مع بياناتهم)
# ============================================

MIN_DATASET_SIZE = 10000  # Minimum target

if len(all_texts) < MIN_DATASET_SIZE:
    repeat_factor = max(2, MIN_DATASET_SIZE // max(len(all_texts), 1))
    print(f"\n⚠️ Dataset is small ({len(all_texts)} texts), repeating {repeat_factor}x for better training...")
    all_texts = all_texts * repeat_factor
    random.shuffle(all_texts)
    print(f"  After augmentation: {len(all_texts)} texts")
else:
    print(f"\n✅ Dataset size sufficient: {len(all_texts)} texts")


✅ Dataset size sufficient: 504565 texts


## 5. تجهيز البيانات للتدريب (Tokenization + MLM)

In [25]:
from datasets import Dataset

# Create HuggingFace Dataset
raw_dataset = Dataset.from_dict({"text": all_texts})

# Split into train/eval
split = raw_dataset.train_test_split(test_size=0.05, seed=SEED)
train_dataset = split["train"]
eval_dataset = split["test"]

print(f"Train size: {len(train_dataset)}")
print(f"Eval size: {len(eval_dataset)}")

Train size: 479336
Eval size: 25229


In [26]:
def tokenize_function(examples):
    """Tokenize texts for MLM training"""
    return tokenizer(
        examples["text"],
        truncation=True,
        max_length=MAX_SEQ_LENGTH,
        padding="max_length",
        return_special_tokens_mask=True,
    )

# Tokenize
print("🔄 Tokenizing training data...")
tokenized_train = train_dataset.map(
    tokenize_function,
    batched=True,
    remove_columns=["text"],
    desc="Tokenizing train",
)

print("🔄 Tokenizing eval data...")
tokenized_eval = eval_dataset.map(
    tokenize_function,
    batched=True,
    remove_columns=["text"],
    desc="Tokenizing eval",
)

print(f"\n✅ Tokenized train: {len(tokenized_train)} samples")
print(f"✅ Tokenized eval: {len(tokenized_eval)} samples")
print(f"📐 Sample keys: {tokenized_train.column_names}")

🔄 Tokenizing training data...


Tokenizing train:   0%|          | 0/479336 [00:00<?, ? examples/s]

🔄 Tokenizing eval data...


Tokenizing eval:   0%|          | 0/25229 [00:00<?, ? examples/s]


✅ Tokenized train: 479336 samples
✅ Tokenized eval: 25229 samples
📐 Sample keys: ['input_ids', 'token_type_ids', 'attention_mask', 'special_tokens_mask']


## 6. Intilaize Training with Unsloth + HuggingFace Trainer



In [27]:
from transformers import (
    DataCollatorForLanguageModeling,
    TrainingArguments,
    Trainer,
)

# MLM Data Collator - handles random masking
data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=True,
    mlm_probability=MLM_PROBABILITY,
)

print("✅ Data collator ready (MLM probability: {:.0%})".format(MLM_PROBABILITY))

✅ Data collator ready (MLM probability: 15%)


In [30]:
# Training Arguments
training_args = TrainingArguments(
    output_dir=f"./{OUTPUT_MODEL_NAME}-checkpoints",
    # overwrite_output_dir=True,

    # Training hyperparameters
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    learning_rate=LEARNING_RATE,
    weight_decay=0.01,
    warmup_ratio=WARMUP_RATIO,

    # Optimizer
    optim="adamw_torch",
    adam_beta1=0.9,
    adam_beta2=0.999,
    adam_epsilon=1e-6,
    max_grad_norm=1.0,

    # Mixed precision for speed
    fp16=torch.cuda.is_available(),

    # Evaluation
    eval_strategy="steps",
    eval_steps=500,

    # Logging
    logging_dir=f"./{OUTPUT_MODEL_NAME}-logs",
    logging_steps=100,
    report_to="none",

    # Saving
    save_strategy="steps",
    save_steps=1000,
    save_total_limit=3,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,

    # Seed
    seed=SEED,

    # Gradient accumulation for effective larger batch
    gradient_accumulation_steps=4,
)

print("✅ Training arguments configured")
print(f"   Effective batch size: {BATCH_SIZE * 4}")
print(f"   Total steps: ~{len(tokenized_train) * NUM_EPOCHS // (BATCH_SIZE * 4)}")

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.
`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


✅ Training arguments configured
   Effective batch size: 128
   Total steps: ~18724


In [31]:
# Initialize Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    data_collator=data_collator,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_eval,
)

print("✅ Trainer initialized")
print(f"\n📋 Training Summary:")
print(f"   Model: {MODEL_NAME}")
print(f"   Task: Masked Language Modeling (MLM)")
print(f"   Train samples: {len(tokenized_train)}")
print(f"   Eval samples: {len(tokenized_eval)}")
print(f"   Epochs: {NUM_EPOCHS}")
print(f"   Learning Rate: {LEARNING_RATE}")
print(f"   Batch Size (effective): {BATCH_SIZE * 4}")

✅ Trainer initialized

📋 Training Summary:
   Model: aubmindlab/bert-base-arabertv02-twitter
   Task: Masked Language Modeling (MLM)
   Train samples: 479336
   Eval samples: 25229
   Epochs: 5
   Learning Rate: 2e-05
   Batch Size (effective): 128


## 7. Start Training

In [32]:
import time

print("🚀 بدء تدريب ShamiBERT...")
print("="*50)

start_time = time.time()

# Train!
train_result = trainer.train()

end_time = time.time()
training_time = end_time - start_time

print(f"\n{'='*50}")
print(f"✅ التدريب اكتمل!")
print(f"⏱️ الوقت: {training_time/60:.1f} دقيقة")
print(f"📉 Train Loss: {train_result.training_loss:.4f}")

🚀 بدء تدريب ShamiBERT...


Step,Training Loss,Validation Loss
500,9.808760,2.286265
1000,9.075778,2.155081
1500,8.579047,2.036978
2000,8.316138,1.938404
2500,8.105450,1.916173
3000,7.851460,1.875267
3500,7.821309,1.850212
4000,7.791474,1.831500
4500,7.619105,1.795138
5000,7.553517,1.809600


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'bert.encoder.layer.3.output.LayerNorm.bias', 'bert.encoder.layer.4.attention.output.La


✅ التدريب اكتمل!
⏱️ الوقت: 129.7 دقيقة
📉 Train Loss: 7.3996


In [33]:
# Evaluate
print("\n📊 تقييم النموذج...")
eval_results = trainer.evaluate()

import math
perplexity = math.exp(eval_results["eval_loss"])

print(f"\n=== نتائج التقييم ===")
print(f"Eval Loss: {eval_results['eval_loss']:.4f}")
print(f"Perplexity: {perplexity:.2f}")


📊 تقييم النموذج...



=== نتائج التقييم ===
Eval Loss: 1.6172
Perplexity: 5.04


## 8. اختبار النموذج بعد التدريب

In [34]:
# Test the trained model with fill-mask
from transformers import pipeline

# Move to eval mode
model.eval()

fill_mask = pipeline("fill-mask", model=model, tokenizer=tokenizer, device=0 if torch.cuda.is_available() else -1)

# Shami test sentences
test_sentences = [
    "كيفك [MASK] الحمدلله",
    "شو [MASK] اليوم",
    "وين [MASK] رايح",
    "هلأ [MASK] بالبيت",
    "يا [MASK] تعال نتعشى سوا",
    "الأكل كتير [MASK]",
    "بدي [MASK] عالسوق",
    "شو [MASK] نعمل بكرا",
    "الجو [MASK] كتير اليوم",
    "ما في [MASK] أحسن من هيك",
]

print("=== اختبار بعد التدريب (After Training) ===")
print("="*50)

for sent in test_sentences:
    try:
        results = fill_mask(sent)
        print(f"\n📝 Input: {sent}")
        for r in results[:3]:
            print(f"   → {r['token_str']:>15} (score: {r['score']:.4f})")
    except Exception as e:
        print(f"   ❌ Error: {e}")

del fill_mask
gc.collect()

=== اختبار بعد التدريب (After Training) ===

📝 Input: كيفك [MASK] الحمدلله
   →               ؟ (score: 0.6202)
   →               ، (score: 0.1538)
   →               - (score: 0.0475)

📝 Input: شو [MASK] اليوم
   →             صار (score: 0.2712)
   →           الطقس (score: 0.0948)
   →            رأيك (score: 0.0877)

📝 Input: وين [MASK] رايح
   →             كنت (score: 0.2622)
   →             انت (score: 0.1788)
   →             ##ك (score: 0.1781)

📝 Input: هلأ [MASK] بالبيت
   →             أنا (score: 0.3117)
   →            قاعد (score: 0.0958)
   →             انا (score: 0.0720)

📝 Input: يا [MASK] تعال نتعشى سوا
   →             عمي (score: 0.2701)
   →           حبيبي (score: 0.1391)
   →             ولد (score: 0.0919)

📝 Input: الأكل كتير [MASK]
   →             طيب (score: 0.2457)
   →             حار (score: 0.1170)
   →             حلو (score: 0.0852)

📝 Input: بدي [MASK] عالسوق
   →             روح (score: 0.7106)
   →            أطلع (score: 0.0666)
   →          

480

## 9. Save Modal Locally

In [35]:
# Save model and tokenizer locally
local_save_path = f"./{OUTPUT_MODEL_NAME}"

print(f"💾 حفظ النموذج في: {local_save_path}")
model.save_pretrained(local_save_path)
tokenizer.save_pretrained(local_save_path)

print("✅ تم حفظ النموذج محلياً")

# Show saved files
import os
for f in os.listdir(local_save_path):
    size = os.path.getsize(os.path.join(local_save_path, f)) / 1e6
    print(f"   📄 {f} ({size:.1f} MB)")

💾 حفظ النموذج في: ./ShamiBERT


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

✅ تم حفظ النموذج محلياً
   📄 config.json (0.0 MB)
   📄 tokenizer.json (1.8 MB)
   📄 tokenizer_config.json (0.0 MB)
   📄 model.safetensors (541.1 MB)


---

## 📝 Important Notes

### Improving the Model
1. **More Data**: Collect more Levantine tweets using the Twitter/X API with location filtering (Syria, Lebanon, Jordan, Palestine).
2. **Forum Data**: Gather data from Levantine forums such as Syrian and Lebanese discussion boards.
3. **Increase Epochs**: With larger datasets, train the model for more epochs.
4. **Vocabulary Expansion**: Add new Levantine words to the tokenizer.
5. **Evaluation**: Use benchmarks such as **ArSenTD-LEV** or **NADI** for evaluation.

### Comparison with SaudiBERT

| Feature | SaudiBERT | ShamiBERT |
|--------|-----------|----------|
| Data | 26.3GB (141M tweets + 70M sentences) | Limited (needs expansion) |
| Base | BERT trained from scratch | Continual pretraining from AraBERT |
| Dialect | Saudi | Levantine (4 dialects) |
| Sources | Twitter + forums | Multiple datasets |

### To Reach SaudiBERT-Level Performance
- Collect **~100M+ Levantine tweets**
- Gather data from **Levantine forums and websites**
- Train a **dedicated Levantine tokenizer**
- Train **from scratch instead of continual pretraining**